In [1]:
# 1. StratifiedKFold 예제 (분류 문제 필수 템플릿)
# •	특징: 정답 라벨($y$)의 클래스 비율을 모든 폴드(Fold)에 원본과 똑같은 비율로 유지하면서 쪼갭니다.언제 쓰나요?: 암 환자 데이터(정상 99%, 암 1%)처럼 특정 클래스가 극단적으로 적은 불균형 데이터 분류 문제를 풀 때, 일반 K-Fold를 쓰면 어떤 폴드에는 암 환자가 단 한 명도 안 들어가는 치명적인 왜곡이 생깁니다. 이를 완벽하게 방지해 줍니다.

import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 데이터 로드 (0~9 손글씨 분류 데이터)
digits = load_digits()
X, y = digits.data, digits.target

# 1. 파이프라인 구성
pipeline_strat = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])

# 2. StratifiedKFold 설정 (5개 폴드, 무작위 섞음)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. 교차 검증 수행
print("--- [1. StratifiedKFold 교차 검증 시작] ---")
strat_scores = cross_val_score(pipeline_strat, X, y, cv=skf, scoring='accuracy', n_jobs=-1)

# 결과 출력
for i, score in enumerate(strat_scores):
    print(f"Fold {i+1} 정확도: {score*100:.2f}%")
print(f"StratifiedKFold 평균 정확도: {np.mean(strat_scores)*100:.2f}%\n")

--- [1. StratifiedKFold 교차 검증 시작] ---
Fold 1 정확도: 96.94%
Fold 2 정확도: 96.67%
Fold 3 정확도: 97.77%
Fold 4 정확도: 97.21%
Fold 5 정확도: 96.94%
StratifiedKFold 평균 정확도: 97.11%



In [2]:
# 2. ShuffleSplit 예제 (유연한 무작위 분할)
# •	특징: 전체 데이터에서 지정한 비율(test_size)만큼 무작위로 떼어내는 과정을 설정한 횟수(n_splits)만큼 반복합니다.
# •	KFold와의 차이점: K-Fold는 중복 없이 전체 데이터를 딱 등분하지만, ShuffleSplit은 무작위 추출이기 때문에 어떤 샘플은 여러 번 검증셋으로 뽑힐 수도 있고, 어떤 샘플은 한 번도 안 뽑힐 수도 있습니다. 또한 폴드 수와 상관없이 훈련/테스트 비율을 내 마음대로 7:3, 8:2 등으로 고정할 수 있어 자유도가 높습니다.

from sklearn.model_selection import ShuffleSplit

# 1. 파이프라인 구성 (동일 모델 사용)
pipeline_shuffle = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])

# 2. ShuffleSplit 설정
# - n_splits=5: 무작위 분할을 총 5번 반복 수행
# - test_size=0.2: 매 반복마다 전체 데이터의 20%를 검증 데이터로 무작위 추출
ss = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

# 3. 교차 검증 수행
print("--- [2. ShuffleSplit 교차 검증 시작] ---")
shuffle_scores = cross_val_score(pipeline_shuffle, X, y, cv=ss, scoring='accuracy', n_jobs=-1)

# 결과 출력
for i, score in enumerate(shuffle_scores):
    print(f"Iteration {i+1} 정확도: {score*100:.2f}%")
print(f"ShuffleSplit 평균 정확도: {np.mean(shuffle_scores)*100:.2f}%")

--- [2. ShuffleSplit 교차 검증 시작] ---
Iteration 1 정확도: 97.22%
Iteration 2 정확도: 98.61%
Iteration 3 정확도: 96.39%
Iteration 4 정확도: 97.22%
Iteration 5 정확도: 97.22%
ShuffleSplit 평균 정확도: 97.33%
